In [ ]:
# source is577/bin/activate
# pip install --upgrade pip
# pip install pandas liac-arff jupyterlab ipykernel
# python -m ipykernel install --user --name=is577 --display-name="Python (is577)"
# pip freeze > requirements.txt
# pip install -r requirements.txt

In [1]:
import pandas as pd
import liac_arff

ModuleNotFoundError: No module named 'liac_arff'

In [2]:
import pandas as pd

def df_to_arff(df: pd.DataFrame, filename: str, relation: str="dataset", nominal_threshold: int=20):
    """
    Write df out to filename in ARFF format.
    
    - Numeric columns become NUMERIC.
    - Object columns with ≤ nominal_threshold unique values become {v1,v2,…}.
    - All other columns become STRING.
    """
    with open(filename, "w", encoding="utf-8") as f:
        # relation
        f.write(f"@RELATION {relation}\n\n")
        
        # attributes
        for col in df.columns:
            series = df[col]
            if pd.api.types.is_numeric_dtype(series):
                f.write(f"@ATTRIBUTE {col} NUMERIC\n")
            else:
                vals = series.dropna().unique().tolist()
                if len(vals) <= nominal_threshold:
                    # escape any commas or braces in values
                    safe = [str(v).replace("'", "\\'") for v in vals]
                    f.write(f"@ATTRIBUTE {col} {{{','.join(safe)}}}\n")
                else:
                    f.write(f"@ATTRIBUTE {col} STRING\n")
        
        # data
        f.write("\n@DATA\n")
        for _, row in df.iterrows():
            out = []
            for v in row:
                if pd.isna(v):
                    out.append('?')
                else:
                    s = str(v)
                    # quote strings containing commas
                    if isinstance(v, str) and "," in s:
                        s = '"' + s.replace('"', '\\"') + '"'
                    out.append(s)
            f.write(",".join(out) + "\n")

In [5]:
df = pd.read_csv("/Users/emine2/Documents/CLASSES/2025-SPRING/IS-577/project/leo_search_results.csv")
df_to_arff(df, "leo_alaska.arff", relation="my_data")